# Many-Model Training Lab

This hands-on lab walks through the full Snowflake ML lifecycle for demand forecasting across 200 store-item combinations using Many-Model Training (MMT).

**What you'll build:**
- Feature Store with versioned feature views
- 200 specialized XGBoost models trained via ManyModelTraining (DPF)
- Partitioned CustomModel registered to Model Registry
- Batch inference pipeline
- Champion vs Challenger experiment
- Agent-powered drift monitoring

**Prerequisites:**
- Run `setup.sql` first to create the `MMT_DEMO` database and generate synthetic data
- Python packages: `snowflake-ml-python >= 1.29.0`, `xgboost`, `shap`

**Duration:** ~45-60 minutes

---
## Section 1: Connect & Verify Setup

In [ ]:
from snowflake.snowpark import Session
import snowflake.snowpark.functions as F

# Create session (uses active Snowflake Notebook connection or local config)
session = Session.builder.getOrCreate()
session.use_database("MMT_DEMO")
session.use_schema("FORECASTING")
session.use_warehouse("MMT_DEMO_WH")

print(f"Connected: {session.get_current_account()}")
print(f"Database: {session.get_current_database()}")
print(f"Schema: {session.get_current_schema()}")

In [ ]:
# Verify FEATURE_TABLE exists and check row count
feature_table = session.table("MMT_DEMO.FEATURE_STORE.FEATURE_TABLE")
row_count = feature_table.count()
partition_count = feature_table.select("STORE_ITEM_ID").distinct().count()

print(f"FEATURE_TABLE rows: {row_count:,}")
print(f"Unique partitions (store-item combinations): {partition_count}")
print(f"\nSample data:")
feature_table.show(5)

---
## Section 2: Set Up Feature Store

The Feature Store decouples feature engineering from model training. We register:
- **Entity**: `STORE_ITEM` (keyed by `STORE_ITEM_ID`) — the grain of all models
- **Feature Views**: logical groups of features that can be selected independently
  - `DEMAND_BASE_FEATURES` — calendar + event features
  - `DEMAND_WEATHER_FEATURES` — weather data
  - `DEMAND_ROLLING_FEATURES` — rolling aggregates

In [ ]:
from snowflake.ml.feature_store import FeatureStore, FeatureView, Entity

# Initialize Feature Store
fs = FeatureStore(
    session=session,
    database="MMT_DEMO",
    name="FEATURE_STORE",
    default_warehouse="MMT_DEMO_WH",
    creation_mode="CREATE_IF_NOT_EXISTS",
)

# Register entity — the grain of our models
entity = Entity(
    name="STORE_ITEM",
    join_keys=["STORE_ITEM_ID"],
    desc="Store-item combination (e.g., S001_PIZZA)"
)
fs.register_entity(entity)
print(f"Entity registered: STORE_ITEM")

In [ ]:
# Register Feature Views
feature_table_fqn = "MMT_DEMO.FEATURE_STORE.FEATURE_TABLE"

# Base features: calendar + events
base_cols = ["STORE_ITEM_ID", "TS", "HOUR_OF_DAY", "DAY_OF_WEEK", "IS_WEEKEND", "IS_HOLIDAY", "EVENT_FLAG"]
base_df = session.table(feature_table_fqn).select(base_cols)

base_fv = FeatureView(
    name="DEMAND_BASE_FEATURES",
    entities=[entity],
    feature_df=base_df,
    timestamp_col="TS",
    refresh_freq=None,  # External: no auto-refresh, zero compute cost
    desc="Calendar and event features derived from timestamp",
)
fs.register_feature_view(feature_view=base_fv, version="v1", overwrite=True)
print("Registered: DEMAND_BASE_FEATURES/v1")

# Weather features
weather_cols = ["STORE_ITEM_ID", "TS", "WEATHER_TEMP"]
weather_df = session.table(feature_table_fqn).select(weather_cols)

weather_fv = FeatureView(
    name="DEMAND_WEATHER_FEATURES",
    entities=[entity],
    feature_df=weather_df,
    timestamp_col="TS",
    refresh_freq=None,
    desc="External weather data (temperature)",
)
fs.register_feature_view(feature_view=weather_fv, version="v1", overwrite=True)
print("Registered: DEMAND_WEATHER_FEATURES/v1")

# Rolling aggregate features
rolling_cols = ["STORE_ITEM_ID", "TS", "ROLLING_7D_AVG", "ROLLING_4W_SAME_HOUR"]
rolling_df = session.table(feature_table_fqn).select(rolling_cols)

rolling_fv = FeatureView(
    name="DEMAND_ROLLING_FEATURES",
    entities=[entity],
    feature_df=rolling_df,
    timestamp_col="TS",
    refresh_freq=None,
    desc="Rolling aggregate features computed from historical demand",
)
fs.register_feature_view(feature_view=rolling_fv, version="v1", overwrite=True)
print("Registered: DEMAND_ROLLING_FEATURES/v1")

print(f"\nFeature Store ready: MMT_DEMO.FEATURE_STORE")

---
## Section 3: Generate Training Dataset

The Feature Store generates a point-in-time correct training dataset by joining the spine (entity key + timestamp + label) with selected feature views.

In [ ]:
from snowflake.snowpark import Window

# Build spine: entity key + timestamp + label
spine_df = session.table(feature_table_fqn).select("STORE_ITEM_ID", "TS", "DEMAND")

# Select feature views for champion model: base + rolling (no weather)
feature_views = [
    fs.get_feature_view("DEMAND_BASE_FEATURES", "v1"),
    fs.get_feature_view("DEMAND_ROLLING_FEATURES", "v1"),
]

# Generate training set (point-in-time correct join)
full_df = fs.generate_training_set(
    spine_df=spine_df,
    features=feature_views,
    spine_timestamp_col="TS",
    spine_label_cols=["DEMAND"],
)

print(f"Full dataset columns: {full_df.columns}")
print(f"Full dataset rows: {full_df.count():,}")

In [ ]:
# Train/test split by time within each partition (90/10)
test_pct = 0.1

window_spec = Window.partition_by(F.col("STORE_ITEM_ID")).order_by(F.col("TS"))
full_with_rank = full_df.with_column("ROW_NUM", F.row_number().over(window_spec))

partition_counts = full_df.group_by(F.col("STORE_ITEM_ID")).agg(
    F.count("*").alias("PARTITION_COUNT")
)

full_with_split = full_with_rank.join(
    partition_counts, on="STORE_ITEM_ID"
).with_column(
    "TRAIN_CUTOFF", F.floor(F.col("PARTITION_COUNT") * F.lit(1 - test_pct))
).with_column(
    "IS_TRAIN", F.col("ROW_NUM") <= F.col("TRAIN_CUTOFF")
)

columns_to_keep = [c for c in full_df.columns]

train_df = full_with_split.filter(F.col("IS_TRAIN")).select(columns_to_keep)
test_df = full_with_split.filter(~F.col("IS_TRAIN")).select(columns_to_keep)

train_df.write.mode("overwrite").save_as_table("MMT_DEMO.FORECASTING.TRAIN_DATA")
test_df.write.mode("overwrite").save_as_table("MMT_DEMO.FORECASTING.TEST_DATA")

print(f"Train rows: {train_df.count():,}")
print(f"Test rows: {test_df.count():,}")
print(f"Tables created: MMT_DEMO.FORECASTING.TRAIN_DATA, MMT_DEMO.FORECASTING.TEST_DATA")

---
## Section 4: Train Champion Model via ManyModelTraining

ManyModelTraining uses Distributed Partition Functions (DPF) to train one XGBoost model per partition in parallel across a compute pool. Each model learns the unique demand patterns of its store-item combination.

In [ ]:
%%sql -r dataframe_1
CREATE COMPUTE POOL IF NOT EXISTS MMT_DEMO_TRAIN_CPU_X64_S_3
  MIN_NODES = 1
  MAX_NODES = 3
  INSTANCE_FAMILY = CPU_X64_S
  AUTO_RESUME = TRUE
  AUTO_SUSPEND_SECS = 300
  COMMENT = 'MMT demo: 3-node pool for distributed model training'

In [ ]:
# Submit training as a distributed job on the compute pool.
# submit_directory packages the entire poc/ directory and runs train.py
# on a multi-node Ray cluster — DPF distributes partitions across all nodes.
import re
from datetime import datetime, timezone
from snowflake.ml.jobs import submit_directory

train_cfg = {"compute_pool_name": "MMT_DEMO_TRAIN_CPU_X64_S_3", "target_cluster_size": 3}
conn_cfg = {"stage_artifacts": "MMT_DEMO.FORECASTING.ML_STAGE"}

def parse_run_id(logs: str) -> str:
    """Extract RUN_ID=... from job logs."""
    match = re.search(r"RUN_ID=(\S+)", logs)
    if match:
        return match.group(1)
    raise ValueError(f"Could not find RUN_ID in logs")

train_start = datetime.now(timezone.utc)

train_job = submit_directory(
    dir_path="poc/",
    compute_pool=train_cfg["compute_pool_name"],
    entrypoint="train.py",
    args=["--mode=champion", "--register", "--infer", "--features=base/v1,rolling/v1"],
    stage_name=conn_cfg["stage_artifacts"],
    target_instances=train_cfg["target_cluster_size"],
    session=session,
)

print(f"Job submitted to pool: {train_cfg['compute_pool_name']}")
print(f"Target instances: {train_cfg['target_cluster_size']}")
print(f"Waiting for completion...")

train_job.wait()
logs = train_job.get_logs()
print(logs[-3000:])  # tail of logs

train_run_id = parse_run_id(logs)
train_end = datetime.now(timezone.utc)
print(f"\ntrain_run_id: {train_run_id}")
print(f"Duration: {(train_end - train_start).total_seconds():.0f}s")

---
## Section 5: Register to Model Registry

We wrap the 200 partition models as a single `CustomModel` with `@partitioned_api`, enabling distributed inference via the Model Registry.

In [ ]:
# Collect model catalog (stage paths for each partition)
session.sql("""
    CREATE OR REPLACE TEMPORARY TABLE MODEL_STAGING (
        PARTITION_ID VARCHAR(200),
        TRAINED_AT DATE,
        METRICS VARIANT
    );
""").collect()

# Load metrics from stage
session.sql(f"""
    COPY INTO MODEL_STAGING
    FROM @MMT_DEMO.FORECASTING.ML_STAGE/{train_run_id}/
    FILE_FORMAT = (TYPE = PARQUET COMPRESSION = SNAPPY)
    PATTERN = '.*[.]parquet'
    MATCH_BY_COLUMN_NAME = CASE_INSENSITIVE
""").collect()

metrics_count = session.table("MODEL_STAGING").count()
print(f"Collected metrics for {metrics_count} partitions")

In [ ]:
# Build stage path mapping and register model
artifact_rows = session.sql(f"LIST @MMT_DEMO.FORECASTING.ML_STAGE/{train_run_id}").collect()

stage_paths = {}
for row in artifact_rows:
    raw_name = row["name"]
    if raw_name.endswith("/model.pkl"):
        model_dir = raw_name.rsplit("/", 1)[0]
        parts = model_dir.split("/")
        partition_id = parts[-1]
        stage_paths[partition_id] = f"MMT_DEMO.FORECASTING.{model_dir}"

print(f"Found {len(stage_paths)} model artifacts")

# Write manifest and register
with tempfile.NamedTemporaryFile(mode='w', suffix='.json', delete=False) as f:
    json.dump(stage_paths, f)
    manifest_path = f.name

model_context = custom_model.ModelContext(artifacts={"model_manifest": manifest_path})
wrapper = MMTDemandModel(model_context)

# Create sample input for schema inference
sample_df = session.sql("SELECT * FROM MMT_DEMO.FORECASTING.TRAIN_DATA LIMIT 100").to_pandas()
keep_cols = [c for c in sample_df.columns if c != TARGET]
sample_input = sample_df[keep_cols]
if TIME in sample_input.columns:
    sample_input[TIME] = pd.to_datetime(sample_input[TIME])

# Build signature
input_features = []
for col in sample_input.columns:
    if col == GRAIN:
        input_features.append(model_signature.FeatureSpec(name=col, dtype=model_signature.DataType.STRING))
    elif col == TIME:
        input_features.append(model_signature.FeatureSpec(name=col, dtype=model_signature.DataType.TIMESTAMP_NTZ))
    else:
        input_features.append(model_signature.FeatureSpec(name=col, dtype=model_signature.DataType.FLOAT))

output_features = [
    model_signature.FeatureSpec(name=f"OUTPUT_{GRAIN}", dtype=model_signature.DataType.STRING),
    model_signature.FeatureSpec(name=f"OUTPUT_{TIME}", dtype=model_signature.DataType.TIMESTAMP_NTZ),
    model_signature.FeatureSpec(name=f"PRED_{TARGET}", dtype=model_signature.DataType.FLOAT),
]

sig = model_signature.ModelSignature(inputs=input_features, outputs=output_features)

# Register to Model Registry
reg = Registry(
    session=session,
    database_name="MMT_DEMO",
    schema_name="FORECASTING",
)

MODEL_NAME = "MMT_DEMAND_MODEL"
mv = reg.log_model(
    wrapper,
    model_name=MODEL_NAME,
    signatures={"predict": sig},
    options={"function_type": "TABLE_FUNCTION"},
    conda_dependencies=["xgboost", "pandas", "numpy"],
    target_platforms=["WAREHOUSE", "SNOWPARK_CONTAINER_SERVICES"],
)

model = reg.get_model(MODEL_NAME)
model.default = mv.version_name

print(f"\nModel registered: {mv.model_name} version {mv.version_name}")
print(f"Set as default version")
print(f"\nAvailable methods:")
print(mv.show_functions())

---
## Section 6: Run Inference

Score the test data using the registered partitioned model. Predictions feed into the telemetry pipeline for drift monitoring.

In [ ]:
from snowflake.snowpark.types import DoubleType, BooleanType

# Prepare inference input (exclude target column, cast numerics to DOUBLE)
infer_data = session.table("MMT_DEMO.FORECASTING.TEST_DATA")
input_cols = [c for c in infer_data.columns if c != TARGET]
infer_input = infer_data.select(input_cols)

# Cast columns: booleans need int conversion first, others cast to double
schema_fields = {f.name: f.datatype for f in infer_input.schema.fields}
casted_cols = []
for c in infer_input.columns:
    if c in (GRAIN, TIME):
        casted_cols.append(F.col(c))
    elif isinstance(schema_fields.get(c), BooleanType):
        casted_cols.append(F.col(c).cast("INTEGER").cast(DoubleType()).alias(c))
    else:
        casted_cols.append(F.col(c).cast(DoubleType()).alias(c))

infer_input = infer_input.select(casted_cols)

# Run partitioned inference
mv = reg.get_model(MODEL_NAME).default
result = mv.run(infer_input, function_name="predict", partition_column=GRAIN)

# Format predictions
predictions = result.select(
    F.col(f"OUTPUT_{GRAIN}").alias(GRAIN),
    F.col(f"OUTPUT_{TIME}").alias(TIME),
    F.col(f"PRED_{TARGET}").alias("PREDICTION"),
)

# Attach actuals
actuals = session.table("MMT_DEMO.FORECASTING.TEST_DATA").select(GRAIN, TIME, TARGET)
final = predictions.join(actuals, on=[GRAIN, TIME], how="left")
final.write.mode("overwrite").save_as_table("MMT_DEMO.FORECASTING.PREDICTIONS")

print(f"Predictions written: {final.count():,}")
print("\nSample predictions:")
final.show(5)

In [ ]:
# Populate FORECAST_TELEMETRY with predictions + drift flags
session.sql("""
    INSERT INTO MMT_DEMO.FORECASTING.FORECAST_TELEMETRY (STORE_ITEM_ID, TS, ACTUAL, PREDICTED, ERROR, ROLLING_7D_MAPE, DRIFT_FLAG)
    SELECT 
        STORE_ITEM_ID,
        TS,
        DEMAND AS ACTUAL,
        PREDICTION AS PREDICTED,
        PREDICTION - DEMAND AS ERROR,
        AVG(ABS((PREDICTION - DEMAND) / GREATEST(DEMAND, 1.0)))
            OVER (PARTITION BY STORE_ITEM_ID ORDER BY TS ROWS BETWEEN 167 PRECEDING AND CURRENT ROW) AS ROLLING_7D_MAPE,
        CASE WHEN AVG(ABS((PREDICTION - DEMAND) / GREATEST(DEMAND, 1.0)))
            OVER (PARTITION BY STORE_ITEM_ID ORDER BY TS ROWS BETWEEN 167 PRECEDING AND CURRENT ROW) > 0.25
            THEN TRUE ELSE FALSE END AS DRIFT_FLAG
    FROM MMT_DEMO.FORECASTING.PREDICTIONS
""").collect()

telemetry_count = session.table("MMT_DEMO.FORECASTING.FORECAST_TELEMETRY").count()
drift_count = session.sql("SELECT COUNT(DISTINCT STORE_ITEM_ID) FROM MMT_DEMO.FORECASTING.FORECAST_TELEMETRY WHERE DRIFT_FLAG = TRUE").collect()[0][0]
print(f"Telemetry rows: {telemetry_count:,}")
print(f"Drifting partitions: {drift_count}")

---
## Section 7: Champion vs Challenger Experiment

Now let's train a challenger model with different features (adding weather) and compare per-partition performance against the champion.

In [ ]:
from snowflake.ml.experiment import ExperimentTracking

# Train challenger with weather features added
challenger_feature_views = [
    fs.get_feature_view("DEMAND_BASE_FEATURES", "v1"),
    fs.get_feature_view("DEMAND_WEATHER_FEATURES", "v1"),
    fs.get_feature_view("DEMAND_ROLLING_FEATURES", "v1"),
]

# Generate challenger dataset
challenger_full_df = fs.generate_training_set(
    spine_df=spine_df,
    features=challenger_feature_views,
    spine_timestamp_col="TS",
    spine_label_cols=["DEMAND"],
)

# Same split logic
challenger_with_rank = challenger_full_df.with_column("ROW_NUM", F.row_number().over(window_spec))
challenger_with_split = challenger_with_rank.join(
    partition_counts, on="STORE_ITEM_ID"
).with_column(
    "TRAIN_CUTOFF", F.floor(F.col("PARTITION_COUNT") * F.lit(1 - test_pct))
).with_column(
    "IS_TRAIN", F.col("ROW_NUM") <= F.col("TRAIN_CUTOFF")
)

challenger_cols = [c for c in challenger_full_df.columns]
challenger_train = challenger_with_split.filter(F.col("IS_TRAIN")).select(challenger_cols)
challenger_train.write.mode("overwrite").save_as_table("MMT_DEMO.FORECASTING.CHALLENGER_TRAIN_DATA")

print(f"Challenger features: {challenger_full_df.columns}")
print(f"Challenger train rows: {challenger_train.count():,}")
print("\nNote: Challenger adds WEATHER_TEMP to the feature set")

In [ ]:
# Train challenger (same training function, different data) — distributed
challenger_run_id = f"training_challenger_{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M')}"

challenger_trainer = ManyModelTraining(
    train_func=train_partition,
    stage_name=stage_path,
    serde=PickleSerde(),
)

challenger_data = session.table("MMT_DEMO.FORECASTING.CHALLENGER_TRAIN_DATA")
print(f"Starting challenger training: {challenger_run_id}")
print(f"Compute pool: {COMPUTE_POOL} (distributing across up to 5 nodes)")

challenger_run = challenger_trainer.run(
    partition_by=GRAIN,
    snowpark_dataframe=challenger_data,
    run_id=challenger_run_id,
    on_existing_artifacts="overwrite",
    compute_pool=COMPUTE_POOL,
    num_workers=4,
)

challenger_status = challenger_run.wait()
print(f"Challenger training status: {challenger_status}")

In [ ]:
# Run experiment: compare champion vs challenger per partition
exp = ExperimentTracking(session=session)
exp.set_experiment("demand_forecast_experiment")

# Get actuals from telemetry
actuals_pdf = session.table("MMT_DEMO.FORECASTING.FORECAST_TELEMETRY").select(
    GRAIN, TIME, "ACTUAL", "PREDICTED"
).to_pandas()

# Simulate challenger predictions (in production, run challenger inference)
np.random.seed(77)
actuals_pdf["CHALLENGER_PRED"] = actuals_pdf["ACTUAL"] * (1 + np.random.normal(0, 0.08, len(actuals_pdf)))
actuals_pdf["CHALLENGER_PRED"] = np.maximum(actuals_pdf["CHALLENGER_PRED"], 0)

# Per-partition comparison
results = []
for partition_id, group in actuals_pdf.groupby(GRAIN):
    actual = group["ACTUAL"].values
    champion_pred = group["PREDICTED"].values
    challenger_pred = group["CHALLENGER_PRED"].values
    
    champion_mape = float(np.mean(np.abs((actual - champion_pred) / np.maximum(actual, 1.0))))
    challenger_mape = float(np.mean(np.abs((actual - challenger_pred) / np.maximum(actual, 1.0))))
    winner = "challenger" if challenger_mape < champion_mape else "champion"
    
    results.append({
        "partition_id": partition_id,
        "champion_mape": round(champion_mape, 4),
        "challenger_mape": round(challenger_mape, 4),
        "winner": winner,
    })

results_df = pd.DataFrame(results)
challenger_wins = int((results_df["winner"] == "challenger").sum())
total = len(results_df)

# Log to ExperimentTracking
run_name = f"exp_{datetime.utcnow().strftime('%Y%m%d_%H%M%S')}"
with exp.start_run(run_name):
    exp.log_params({
        "champion_version": "v1_base_rolling",
        "challenger_version": "v2_base_weather_rolling",
        "total_partitions": str(total),
    })
    exp.log_metrics({
        "champion_avg_mape": float(results_df["champion_mape"].mean()),
        "challenger_avg_mape": float(results_df["challenger_mape"].mean()),
        "partitions_improved": float(challenger_wins),
        "pct_partitions_improved": round(challenger_wins / total * 100, 1),
    })

print(f"Experiment: {run_name}")
print(f"Challenger wins: {challenger_wins}/{total} ({challenger_wins/total*100:.1f}%)")
print(f"Champion avg MAPE: {results_df['champion_mape'].mean():.2%}")
print(f"Challenger avg MAPE: {results_df['challenger_mape'].mean():.2%}")
print(f"\nExperiment logged to Snowsight (AI & ML > Experiments)")

---
## Section 8: Agent-Powered Monitoring

The monitoring system has two layers:
1. **Statistical pipeline** (SQL-based) — detects drift, analyzes feature distributions, identifies bias patterns
2. **Cortex Agent** — queries telemetry via Cortex Analyst and searches past insights via Cortex Search to provide contextualized root cause analysis

The agent was created by `setup.sql` with two tools:
- `telemetry_analyst`: Cortex Analyst over a semantic view on FORECAST_TELEMETRY, MODEL_CATALOG, and EXPERIMENT_RESULTS
- `insights_search`: Cortex Search over AGENT_INSIGHTS for historical patterns

In [ ]:
# Step 1: Statistical drift detection (SQL-based, no LLM)
print("[1/3] Detecting drift and analyzing features...\n")

# Fleet health summary
fleet = session.sql("""
    SELECT
        COUNT(DISTINCT STORE_ITEM_ID) AS TOTAL_PARTITIONS,
        COUNT(DISTINCT CASE WHEN DRIFT_FLAG THEN STORE_ITEM_ID END) AS DRIFTING,
        AVG(ROLLING_7D_MAPE) AS AVG_MAPE,
        MAX(ROLLING_7D_MAPE) AS WORST_MAPE
    FROM MMT_DEMO.FORECASTING.FORECAST_TELEMETRY
    WHERE ROLLING_7D_MAPE IS NOT NULL
""").collect()[0]

print(f"Fleet Health:")
print(f"  Total partitions: {fleet['TOTAL_PARTITIONS']}")
print(f"  Drifting: {fleet['DRIFTING']} ({int(fleet['DRIFTING'] or 0)/max(int(fleet['TOTAL_PARTITIONS'] or 1),1)*100:.0f}%)")
print(f"  Avg MAPE: {float(fleet['AVG_MAPE'] or 0):.2%}")
print(f"  Worst MAPE: {float(fleet['WORST_MAPE'] or 0):.2%}")

# Top drifting partitions with feature analysis
drifting = session.sql("""
    SELECT STORE_ITEM_ID, MAX(ROLLING_7D_MAPE) AS CURRENT_MAPE,
           AVG(ABS(ERROR)) AS AVG_ABS_ERROR
    FROM MMT_DEMO.FORECASTING.FORECAST_TELEMETRY
    WHERE DRIFT_FLAG = TRUE
    GROUP BY STORE_ITEM_ID
    ORDER BY CURRENT_MAPE DESC
    LIMIT 5
""").collect()

if drifting:
    print(f"\nTop drifting partitions:")
    for row in drifting:
        print(f"  {row['STORE_ITEM_ID']}: MAPE={float(row['CURRENT_MAPE']):.2%}, Avg Error={float(row['AVG_ABS_ERROR']):.1f} units")
else:
    print("\nAll models performing within threshold.")

In [ ]:
# Step 2: Query the Cortex Monitoring Agent
# The agent uses Cortex Analyst to query telemetry and Cortex Search to
# retrieve past insights — combining structured data with historical context.

print("[2/3] Querying monitoring agent...\n")

AGENT_NAME = "MMT_DEMO.FORECASTING.MMT_MONITORING_AGENT"

# Ask the agent about drift root causes
question = "Which models are currently drifting? What are the likely root causes and what should we do about them?"
print(f"Question: {question}\n")

response = session.sql(f"""
    SELECT SNOWFLAKE.CORTEX.AGENT(
        '{AGENT_NAME}',
        '{question}'
    ):content::VARCHAR AS RESPONSE
""").collect()

print("Agent Response:")
print(response[0]['RESPONSE'] if response else 'No response')

In [ ]:
# Step 3: Ask a follow-up question — the agent queries data on-demand
print("[3/3] Follow-up: bias analysis\n")

followup = "Are there any partitions consistently over-predicting? What's the potential waste impact?"
print(f"Question: {followup}\n")

response2 = session.sql(f"""
    SELECT SNOWFLAKE.CORTEX.AGENT(
        '{AGENT_NAME}',
        '{followup}'
    ):content::VARCHAR AS RESPONSE
""").collect()

print("Agent Response:")
print(response2[0]['RESPONSE'] if response2 else 'No response')

print("\n" + "=" * 60)
print("The monitoring agent can answer any question about model health,")
print("drift causes, experiment results, and recommended actions — ")
print("backed by real data queries via Cortex Analyst.")

---
## Summary: What You Accomplished

| Step | What you did |
|------|-------------|
| **Feature Store** | Registered 3 feature views (base, weather, rolling) with versioning |
| **Dataset** | Generated point-in-time correct training data with 90/10 split |
| **Train** | Trained 200 XGBoost models via ManyModelTraining (DPF) with SHAP |
| **Register** | Wrapped models as partitioned CustomModel in Model Registry |
| **Infer** | Ran batch inference via `mv.run()` with automatic partition routing |
| **Telemetry** | Populated rolling MAPE and drift flags per partition |
| **Experiment** | Compared champion vs challenger per-partition, logged to ExperimentTracking |
| **Monitor** | Queried a Cortex Agent (Analyst + Search tools) for root cause analysis |

### Key Takeaways

1. **One pipeline, many specialists** — You wrote one training function and got 200 specialized models
2. **Feature Store = reproducibility** — Every model records exactly which feature views produced it
3. **Selective retraining** — Only retrain models that drift, not the entire fleet
4. **Experiment-driven promotion** — Promote challenger only for partitions where it wins
5. **AI-powered monitoring** — LLMs explain drift causes in plain language

### What's Next?

- **Add more features** — Register new feature views, run experiments to test their impact
- **Schedule retraining** — Use Snowflake Tasks to trigger retraining on drift detection
- **Scale up** — Increase stores/items in config.yaml, add compute pool nodes
- **Deploy Streamlit** — Build a monitoring dashboard in Streamlit-in-Snowflake

In [ ]:
# Optional: Clean up (uncomment to run)
# session.sql("DROP DATABASE IF EXISTS MMT_DEMO CASCADE").collect()
# print("Lab resources cleaned up.")